In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, accuracy_score,  recall_score, f1_score


In [6]:
# set display options for pandas and seaborn
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option("display.float_format", lambda x: "%.3f" % x)
sns.set_theme(style="darkgrid")

csv_file_path = "../dataset/diabetes.csv"

In [7]:
df = pd.read_csv(csv_file_path)
print(df.head())

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin    BMI  \
0            6      148             72             35        0 33.600   
1            1       85             66             29        0 26.600   
2            8      183             64              0        0 23.300   
3            1       89             66             23       94 28.100   
4            0      137             40             35      168 43.100   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  


In [8]:
df.shape

(768, 9)

In [10]:
#EDA
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [11]:
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [13]:
# quick check encoded missing values
for col in df.columns:
    print(df[col].value_counts().head(5))

Pregnancies
1    135
0    111
2    103
3     75
4     68
Name: count, dtype: int64
Glucose
99     17
100    17
111    14
125    14
129    14
Name: count, dtype: int64
BloodPressure
70    57
74    52
78    45
68    45
72    44
Name: count, dtype: int64
SkinThickness
0     227
32     31
30     27
27     23
23     22
Name: count, dtype: int64
Insulin
0      374
105     11
130      9
140      9
120      8
Name: count, dtype: int64
BMI
32.000    13
31.600    12
31.200    12
0.000     11
32.400    10
Name: count, dtype: int64
DiabetesPedigreeFunction
0.258    6
0.254    6
0.207    5
0.261    5
0.259    5
Name: count, dtype: int64
Age
22    72
21    63
25    48
24    46
23    38
Name: count, dtype: int64
Outcome
0    500
1    268
Name: count, dtype: int64


In [14]:
# identify columns with zero values that may represent missing data
zero_summary =(
    df.eq(0).sum().to_frame(name='zero_count').assign(zero_percentage=lambda x: (x['zero_count'] / len(df)) * 100)
)

print(zero_summary)

                          zero_count  zero_percentage
Pregnancies                      111           14.453
Glucose                            5            0.651
BloodPressure                     35            4.557
SkinThickness                    227           29.557
Insulin                          374           48.698
BMI                               11            1.432
DiabetesPedigreeFunction           0            0.000
Age                                0            0.000
Outcome                          500           65.104
